In [6]:
#define system variables

import os
from pathlib import Path
from dotenv import find_dotenv
from dotenv import load_dotenv

path_to_env = os.path.join(
    Path().absolute().parent.parent,
    ".env"
)
load_dotenv(dotenv_path=path_to_env)
    
# Get secrets
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_COLLECTION = os.getenv("DB_COLLECTION")
DB_APP_NAME = os.getenv("DB_APP_NAME")
DB_NAME = os.getenv("DB_NAME")

In [7]:
import chatgenie as cg

# building base units
llm = cg.LLM(
    llm_type = cg.LLMTypes.OPENAI,
    template = cg.PromptTemplate.DEFAULT_PROMPT_WITH_HISTORY,
    model = "gpt-4o",
)

embedder = cg.Embedder(
    embedder_type = cg.EmbedderTypes.OPENAI,
)

vector_db = cg.VectorDB(
    db_type = cg.VectorDBTypes.MONGO,
    collection_name= DB_COLLECTION,
    username= DB_USERNAME,
    password= DB_PASSWORD,   
    app_name= DB_APP_NAME, 
    dbname= DB_NAME, 
    dimensions= 1536,
    allow_reset= True,
)

# building specialized agents (this case only one agent)
agent = cg.agent.GeneralRAG(
    llm = llm,
    embedder = embedder,
    db = vector_db,
    max_results = 5,
)

# building communicator which will be used to orchestrate agent to agent communication
# this component handles memory for now (aggregates the memory and pass down to agents 
# for them to specifically handle the way they want)
communicator = cg.communicator.GeneralRAG(
    agent = agent
)

# building thread handler which will be used to handle threading for the agents
# one communicator per thread for now
threadhandler = cg.thread_handler.DictThreadHandler(
    communicator
)

Successfully connected to MongoDB!


### A single communicator

In [8]:
req, res = communicator.query(
            "How do I create a FitSmiles account?"
            )

print(res)

ic| messages: [HumanMessage(content="
                Use the following pieces of context to answer the query at the end.
                If the question is not related to obesity or weight-loss, just say that you don't know, don't try to make up an answer.
                I will provide you with our conversation history.
              
                How do I create a FitSmiles account? | How do I delete my FitSmiles account? | How do I set a personal goal in FitSmiles? | How do I connect a smartwatch to FitSmiles? | How do track my FitSmiles history? | How can I earn more FitSmiles points following the FitSmiles

data_list [{'_id': ObjectId('673abaf743b97b76f960d7fc'), 'text': 'How do I create a FitSmiles account?', 'metadata': {'url': 'local', 'answer': 'Step 1: Download the FitSmiles Mobile App Step 2: Open the app and explore its main features and functionality. As you go through each screen, click "Next" to proceed Once you\'ve reached the end of the instructions, click on the "Get Started" button, then you\'ll be redirected to the register page. From there, you can create a FitSmiles account. Step 3: Enter your First Name, Last Name, Email, and Password and Click on the "Register" Button.Special Note*: The password must contain at least one uppercase letter, one lowercase letter, and one number, and be at least 8 characters long. These criteria are displayed and checked off as the user types.If you are already a FitSmiles member, click on the "Already a member? Sign In" button.', 'data_type': 'qna_pair', 'doc_id': 'mongo_test_app--ec1729598fa1ec159f66a7653faa0a132d843a582ed46c21ca5682914be

 social media? | How do I manage notifications in FitSmiles? | How do I update my personal goal in FitSmiles? | FAQs User Onboarding How do I create a FitSmiles account? Provide a step-by-step guide on creating an account with images. How do I sign in using email and password? How do I sign in using Google? How do I sign in using Apple? How do I reset my password? How do I sign in using FaceID/Fingerprint? Why do I want to “Allow Notification” for FitSmiles? Enable notifications to stay updated on your progress and get reminders to keep smiling! Why do I want to “Allow Location” for FitSmiles? FitSmiles needs access to your location to track your steps, and show nearby partnered locations. Your data stays private and is used only to provide a better app experience. Why do I want to “Change to Always Allow” location for FitSmiles? Enable background location access and set it to 'Always Allow'. This allows us to accurately track your location and steps. Why do I want to “Allow Motion & F

To create a FitSmiles account, you can follow these steps:

1. Download the FitSmiles app from the App Store or Google Play Store.
2. Open the app and click on the "Sign Up" button.
3. Enter your email address and create a password for your account.
4. Follow the on-screen instructions to complete the registration process.
5. Once your account is created, you can set up your personal goals and start earning FitSmiles points by tracking your activity.


### Utilizing Thread Handler

In [9]:
response = threadhandler.request(
            "How do I create a FitSmiles account?",
            agent_id = None
            )

print(response)

ic| messages: [HumanMessage(content="
                Use the following pieces of context to answer the query at the end.
                If the question is not related to obesity or weight-loss, just say that you don't know, don't try to make up an answer.
                I will provide you with our conversation history.
              
                How do I create a FitSmiles account? | How do I delete my FitSmiles account? | How do I set a personal goal in FitSmiles? | How do I connect a smartwatch to

data_list [{'_id': ObjectId('673abaf743b97b76f960d7fc'), 'text': 'How do I create a FitSmiles account?', 'metadata': {'url': 'local', 'answer': 'Step 1: Download the FitSmiles Mobile App Step 2: Open the app and explore its main features and functionality. As you go through each screen, click "Next" to proceed Once you\'ve reached the end of the instructions, click on the "Get Started" button, then you\'ll be redirected to the register page. From there, you can create a FitSmiles account. Step 3: Enter your First Name, Last Name, Email, and Password and Click on the "Register" Button.Special Note*: The password must contain at least one uppercase letter, one lowercase letter, and one number, and be at least 8 characters long. These criteria are displayed and checked off as the user types.If you are already a FitSmiles member, click on the "Already a member? Sign In" button.', 'data_type': 'qna_pair', 'doc_id': 'mongo_test_app--ec1729598fa1ec159f66a7653faa0a132d843a582ed46c21ca5682914be

 FitSmiles? | How do track my FitSmiles history? | How can I earn more FitSmiles points following the FitSmiles social media? | How do I manage notifications in FitSmiles? | How do I update my personal goal in FitSmiles? | FAQs User Onboarding How do I create a FitSmiles account? Provide a step-by-step guide on creating an account with images. How do I sign in using email and password? How do I sign in using Google? How do I sign in using Apple? How do I reset my password? How do I sign in using FaceID/Fingerprint? Why do I want to “Allow Notification” for FitSmiles? Enable notifications to stay updated on your progress and get reminders to keep smiling! Why do I want to “Allow Location” for FitSmiles? FitSmiles needs access to your location to track your steps, and show nearby partnered locations. Your data stays private and is used only to provide a better app experience. Why do I want to “Change to Always Allow” location for FitSmiles? Enable background location access and set it to

{'request_text': 'How do I create a FitSmiles account?', 'agent_id': 139821816875088, 'response': (<chatgenie.data.query.QueryDataPacket object at 0x7f2a953af090>, 'To create a FitSmiles account, you can follow these steps:\n\n1. Download the FitSmiles app from the App Store or Google Play Store.\n2. Open the app and click on the "Sign Up" button.\n3. Enter your email address and create a password for your account.\n4. Follow the on-screen instructions to complete the registration process.\n5. You may be asked to set a personal goal during the onboarding process.\n6. Once your account is created, you can')}


In [10]:
response = threadhandler.request(
            "How do I create a FitSmiles account?",
            agent_id = None
            )
        
agent_id = response['agent_id']

response = threadhandler.request(
    "What was my previous question about",
    agent_id = agent_id
    )

print(response)

ic| messages: [HumanMessage(content="
                Use the following pieces of context to answer the query at the end.
                If the question is not related to obesity or weight-loss, just say that you don't know, don't try to make up an answer.
                I will provide you with our conversation history.
              
                How do I create

data_list [{'_id': ObjectId('673abaf743b97b76f960d7fc'), 'text': 'How do I create a FitSmiles account?', 'metadata': {'url': 'local', 'answer': 'Step 1: Download the FitSmiles Mobile App Step 2: Open the app and explore its main features and functionality. As you go through each screen, click "Next" to proceed Once you\'ve reached the end of the instructions, click on the "Get Started" button, then you\'ll be redirected to the register page. From there, you can create a FitSmiles account. Step 3: Enter your First Name, Last Name, Email, and Password and Click on the "Register" Button.Special Note*: The password must contain at least one uppercase letter, one lowercase letter, and one number, and be at least 8 characters long. These criteria are displayed and checked off as the user types.If you are already a FitSmiles member, click on the "Already a member? Sign In" button.', 'data_type': 'qna_pair', 'doc_id': 'mongo_test_app--ec1729598fa1ec159f66a7653faa0a132d843a582ed46c21ca5682914be

 a FitSmiles account? | How do I delete my FitSmiles account? | How do I set a personal goal in FitSmiles? | How do I connect a smartwatch to FitSmiles? | How do track my FitSmiles history? | How can I earn more FitSmiles points following the FitSmiles social media? | How do I manage notifications in FitSmiles? | How do I update my personal goal in FitSmiles? | FAQs User Onboarding How do I create a FitSmiles account? Provide a step-by-step guide on creating an account with images. How do I sign in using email and password? How do I sign in using Google? How do I sign in using Apple? How do I reset my password? How do I sign in using FaceID/Fingerprint? Why do I want to “Allow Notification” for FitSmiles? Enable notifications to stay updated on your progress and get reminders to keep smiling! Why do I want to “Allow Location” for FitSmiles? FitSmiles needs access to your location to track your steps, and show nearby partnered locations. Your data stays private and is used only to provi

data_list [{'_id': ObjectId('673abe844880d2938b9a8451'), 'text': 'How do I reset my password?', 'metadata': {'url': 'local', 'answer': 'If you\'re on the login screen, tap on the "Forgot Password?" link to open the password reset interface.\nStep 01: In the "Email address" field, type the email address associated with your FitSmiles account.\nStep 02: Tap the red "Send reset link" button and you will receive a password reset link to the email inbox you Provided\nOpen your email inbox and look for an email from FitSmiles with instructions on how to reset your password.\nFollow the link provided in the email to complete the password reset process.\nOnce your password has been reset, go back to the login screen, enter your new password, and tap "Sign in" to access your account.', 'data_type': 'qna_pair', 'doc_id': 'mongo_test_app--ec38a3efd43a57771189e85ad6355a36c41696e951ebe4aad09ab9977826e241', 'app_id': 'mongo_test_app', 'hash': '5373b756336d7056096b8f61fe7029b9', 'attributes': ['fitsm

 I edit my profile details? | How do I sign in using email and password? | How do I manage my privacy and security settings?
              
                History: Human: How do I create a FitSmiles account?
              AI: To create a FitSmiles account, you can follow these steps:
              
              1. Download the FitSmiles app from the App Store or Google Play Store.
              2. Open the app and click on the "Sign Up" or "Create Account" button.
              3. Enter your email address and create a password for your account.
              4. Follow the on-screen instructions to complete the registration process, which may include entering your name, age, gender, and other relevant information.
              5. Once you have successfully
              Human: What was my previous question about
              
                Query: <chatgenie.data.query.QueryDataPacket object at 0x7f2acdb78850>
              
                Helpful Answer:
              ', addition

{'request_text': 'What was my previous question about', 'agent_id': 139820849736080, 'response': (<chatgenie.data.query.QueryDataPacket object at 0x7f2acdb78850>, 'Your previous question was about creating a FitSmiles account.')}
